<a href="https://colab.research.google.com/github/MouseLand/cellpose/blob/main/notebooks/run_Cellpose-SAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Run Cellpose-SAM - Stitched Image Version

Adapted from Marius Pachitariu, Michael Rariden, Carsen Stringer and the notebook by Pradeep Rajasekhar, inspired by the [ZeroCostDL4Mic notebook series](https://github.com/HenriquesLab/ZeroCostDL4Mic/wiki)

Designed for stitched global 5x4 2D images of 20 2160x20160 fields (roughly 10800x8140)

[paper](https://www.biorxiv.org/content/10.1101/2025.04.28.651001v1) | [code](https://github.com/MouseLand/cellpose)

### Make sure you are in the correct environment


In [ ]:
# Check GPU and instantiate model - will download weights.
import numpy as np
from cellpose import models, core, io, plot, utils
from pathlib import Path
from tqdm import trange
import matplotlib.pyplot as plt
import cv2 as cv 
import tifffile as tf
%matplotlib inline
from natsort import natsorted
from skimage import exposure, filters, morphology
from cellpose_functions import *
import pandas as pd


io.logger_setup() # run this to get printing of progress

#Check if GPU access


In [ ]:
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime")

model = models.CellposeModel(gpu=True)

Input directory with your images:
- Note - For best accuracy and runtime performance, resize images so cells are less than 100 pixels across

### set up the correct order for the image filenames - sort by location first, then channel
these functions moved to cellpose_functions
```python
def file_sort_key(filename):
  parts = filename.split("-")
  channel = parts[0][-1:] # get the last character of the first part
  location = parts[1]
  return (location,channel)

def plate_location(filename):
  parts = filename.split("-")
  pre_location = parts[1]
  location = pre_location.split(".")[0] # get the first part of the second part
  return location
  
#list all files
def sort_files(dir, image_ext):
  if not dir.exists():
    raise FileNotFoundError("directory does not exist")
  files = sorted([f for f in dir.glob("*"+image_ext) if "_masks" not in f.name and "_flows" not in f.name and "SUM" not in f.name],
                           key=lambda x: file_sort_key(x.name))```
 # sort by number in filename
  if(len(files)==0):
    raise FileNotFoundError("no image files found, did you specify the correct folder and extension?")
  else:
    return files
  
def print_files(files):
  for f in files:
    print(f.name)

def group_files_by_channel(files, nchannels=4):
  grouped = []
  for i in range(0,len(files),nchannels):
    grouped.append(files[i:i+nchannels])
  return grouped

def print_grouped_files(grouped):
  for i in range(len(grouped)):
    print(f"\n Group {i+1} of {len(grouped)}")
    for j in range(len(grouped[i])):
      item = grouped[i][j]
      print(" "+ item.name)
```

In [ ]:
## For stitched images
image_ext = ".ome.tif"

stitched_path = Path("/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/stitching/segmentation_testset/")
stitched_path = Path(stitched_path)
if not stitched_path.exists():
  raise FileNotFoundError("directory does not exist")

# list all files
stitched_files = natsorted([f for f in stitched_path.glob("**/*"+image_ext) if "_masks" not in f.name and "_flows" not in f.name])
for f in stitched_files:
  print(f.name)
  #Make mask directory in the parent folder
  maskdir = f.parent / "masks"
  maskdir.mkdir(exist_ok=True)
  print(maskdir)
  
if(len(stitched_files)==0):
  raise FileNotFoundError("no image files found, did you specify the correct folder and extension?")
else:
  print(f"{len(stitched_files)} images in folder:")





## Run Cellpose-SAM on one image in folder

Here are some of the parameters you can change:

* ***flow_threshold*** is  the  maximum  allowed  error  of  the  flows  for  each  mask.   The  default  is 0.4.
    *  **Increase** this threshold if cellpose is not returning as many masks as you’d expect (or turn off completely with 0.0)
    *   **Decrease** this threshold if cellpose is returning too many ill-shaped masks.

* ***cellprob_threshold*** determines proability that a detected object is a cell.   The  default  is 0.0.
    *   **Decrease** this threshold if cellpose is not returning as many masks as you’d expect or if masks are too small
    *   **Increase** this threshold if cellpose is returning too many masks esp from dull/dim areas.

* ***tile_norm_blocksize*** determines the size of blocks used for normalizing the image. The default is 0, which means the entire image is normalized together.
  You may want to change this to 100-200 pixels if you have very inhomogeneous brightness across your image.
### Channel Selection
If you have a fluroescent image with multiple stains, you should choose one channel with a cytoplasm/membrane stain, one channel with a nuclear stain, and set the third channel to None. Choosing multiple channels may produce segmentaiton of all the structures in the image. If you have retrained the model on your data with a thrid stain (described below), you can run segmentation with all channels.

\n if you want to combine two stains to create your "cytoplasm" channel:
- Combine indices 0 and 2 (1st and 3rd) with two cellular stains
- and nuclei are in index 3 (4th channel)


In [ ]:
first_channel = '0' # @param ['None', 0, 1, 2, 3, 4, 5]
second_channel = '1' # @param ['None', 0, 1, 2, 3, 4, 5]
third_channel = '3' # @param ['None', 0, 1, 2, 3, 4, 5]
channel_axis = 0

channels =  ["LAMP1",
            "MitoTracker",
            "Phalloidin",
            "DAPI"]
#channels.index("LAMP1")
index = 7
# If using a format with ch as the last dimension, Initialize with zeros for two channels

In [ ]:
stitched_img = io.imread(stitched_files[index])
print(f'your image has shape: {stitched_img.shape}. Assuming channel dimension is first with {stitched_img.shape[0]} channels')

def select_channels(stitched_img, first_channel=first_channel, second_channel=second_channel, third_channel=third_channel, show = False):
  selected_channels = []
  #For a multichannel stitched image in n,x,y format
  channel_axis = 0
  for i, c in enumerate([first_channel, second_channel, third_channel]):
    if c == 'None':
      continue
    if int(c) > stitched_img.shape[channel_axis]:
      assert False, 'invalid channel index, must have index greater or equal to the number of channels'
    if c != 'None':
      selected_channels.append(int(c))

  pre_used_channel_img = np.stack((stitched_img[selected_channels[0]], stitched_img[selected_channels[1]], stitched_img[selected_channels[2]]), axis=channel_axis)
  #pre_used_channel_img = np.stack(((stitched_img[0]+stitched_img[1]), stitched_img[channels.index("DAPI")]), axis=channel_axis)
  print(pre_used_channel_img.dtype)
  #tf.imshow(img_channel)
  if show:
    tf.imshow(pre_used_channel_img[[0,1,2]].sum(axis=channel_axis, dtype=np.uint16))
  return pre_used_channel_img

#set_name = get_image_set_name(stitched_files)
#print("Set name: ", set_name)
#display(in_channels)

def get_tif_image_info(img_path):
    tif = tf.TiffFile(img_path)
    npages = len(tif.pages) # number of pages in the file
    page = tif.pages[0]  # get shape and dtype of image in first page
    dtype=page.dtype
    axes = page.axes

    series = tif.series[0]  # get shape and dtype of first image series
    nseries = len(tif.series)
    series_shape = series.shape
    series_type = series.dtype
    series_axes = series.axes
    series_pyramid_levels = len(series.levels)

    ome_metadata = tif.ome_metadata
    info = f"ImageName: {tif.filename} \n pages: {npages} \n dtype: {dtype} \n page axes: {axes} \n number of series: {nseries} \n shape: {series_shape} \n format: {series_axes} \n number of pyramid levels: {series_pyramid_levels} \n series dtype: {series_type}"
    tif.close()
    return info

print(get_tif_image_info(stitched_files[index]))
pre_used_channel_img = select_channels(stitched_img, show=True)

#NOTE: for pyramids, use level in the imread reader for tifffile
# first_level = (tf.imread(stitched_files[0], series=0, level=0))
# second_level = (tf.imread(stitched_files[0], series=0, level=1))
# print(first_level.shape)
# print(second_level.shape)

'''
with TiffFile('temp.ome.tif') as tif:
...     baseimage = tif.series[0].asarray()
...     second_level = tif.series[0].levels[1].asarray()
...     number_levels = len(tif.series[0].levels)  # includes base level
'''


In [ ]:
#Otherwise, group files by channel first
def file_sort_key(filename):
  '''
    Generate a key to sort a list of image files in the 'MAX_chN-rXXcYYfZZ.tif' filename nomenclature by their plate location and channel.
    > e.g: MAX_ch1-r02c02f01.tif, MAX_ch2-r02c02f01.tif, MAX_ch3-r02c02f01.tif, MAX_ch3-r02c02f01.tif, MAX_ch1-r02c02f02.tif, MAX_ch2-r02c02f02.tif, MAX_ch3-r02c02f02.tif, MAX_ch3-r02c02f02.tif
    Parameters:
          filename (str): the filename from the image path
    Returns:
          (location,channel) (tuple of str): the list of files sorted by location and channel. Images will be ordered by location first, and then by channel to match the order seen in CellProfiler  
  '''
  parts = filename.split("-")
  channel = parts[0][-1:] # get the last character of the first part for the channel number
  location = parts[1] #get the rXXcYYfZZ.tif portion
  return (location,channel)

def sort_files(dir, image_ext):
  '''
    Sort the list of directories sorted by their plate location and channel 
    Parameters:
          dir (Path object or str): the directory containing the images
          image_ext (str, optional): the image extension, tif by default
    Returns:
          files (list of Path objects): the list of files sorted by location and channel  
  '''
  if not dir.exists():
    raise FileNotFoundError("directory does not exist")
  files = sorted([f for f in dir.glob("*"+image_ext) if "_masks" not in f.name and "_flows" not in f.name and "SUM" not in f.name],
                           key=lambda x: file_sort_key(x.name))
 # sort by number in filename
  if(len(files)==0):
    raise FileNotFoundError("no image files found, did you specify the correct folder and extension?")
  else:
    return files

def group_files_by_channel(files, nchannels=None):
  '''
    Load the list of directories
    Parameters:
          dir (Path object or str): the directory containing the images
          nchannels (int,optionsl): the number of image channels to use for grouping
    Returns:
          grouped_files_by_channel (2D list of Path objects): list of files grouped into image sets by their channels
  '''
  grouped_files_by_channel = []
  if nchannels==None:
    nchannels = get_nchannels(files)
    
  for i in range(0,len(files),nchannels):
    grouped_files_by_channel.append(files[i:i+nchannels])
  return grouped_files_by_channel

In [ ]:
def preprocess_stitched_img(pre_used_channel_img, channel_axis=0):
    og_img_shape = pre_used_channel_img.shape
    print(og_img_shape)
    #tf.imshow(stitched_img[3,:,:])

    #preprocessed_stitch = np.stack([np.zeros_like(stitched_img[0]), np.zeros_like(stitched_img[0])], axis=-1) 
    #convert image to the (height, width, channels) shape for scikit-image functions
    if og_img_shape[0] < 5:
        used_channel_img = np.transpose(pre_used_channel_img, (1,2,0)) 
        print(used_channel_img.shape)
        channel_axis =-1
    else:
        used_channel_img = pre_used_channel_img
        

    preprocessed_image_channels = []
    for i in range(3):#range(used_channel_img.shape[-1]):
        img_channel = used_channel_img[:,:,i]
        if img_channel is None:
            print(f"Channel {i} is None, skipping.")
            continue
        #footprint = morphology.disk(5)
        #channel = img_01_normalization(channel)
        img_channel = img_channel.astype(float)
        img_channel = img_channel / img_channel.max()
        
        dog = filters.difference_of_gaussians(img_channel, low_sigma=2.5)
        img_channel = img_channel - dog
        img_channel = morphology.closing(img_channel, morphology.disk(3))
        
        #amplify the mito channel 
        if i == 1:
            img_channel = img_channel*2
            img_channel = preprocessed_image_channels[0] + img_channel
            
        img_channel = img_channel.astype(float)
        img_channel = img_channel / img_channel.max()
        
            
        img_channel = exposure.equalize_adapthist(img_channel, kernel_size=50, clip_limit=0.02)
        img_channel = filters.gaussian(img_channel, sigma=2) 
        
        #tf.imshow(img_channel, cmap="gray")  
        preprocessed_image_channels.append(img_channel)
        
    print(preprocessed_image_channels[0].shape)
        
    preprocessed_stitched_img = np.stack((preprocessed_image_channels[1],preprocessed_image_channels[2]), axis=-1)
    return preprocessed_stitched_img

# preprocessed_stitched_img = np.zeros_like(used_channel_img)
# for i in range(used_channel_img.shape[-1]):
#     img_channel = used_channel_img[:,:,i].copy()
#     if img_channel is None:
#         print(f"Channel {i} is None, skipping.")
#         continue
#     #img_channel = img_01_normalization(img_channel)
#     img_channel = exposure.equalize_adapthist(img_channel, kernel_size=100, clip_limit=0.05)
#     img_channel = filters.gaussian(img_channel, sigma=2)
    
#     preprocessed_stitched_img[:,:,i] = img_channel
preprocessed_stitched_img = preprocess_stitched_img(pre_used_channel_img)
print("Preprocessed stitched image shape:", preprocessed_stitched_img.shape)
rescaled_stitch = img_rescaled(preprocessed_stitched_img, factor=0.25, channel_axis=-1)
#img_channel = img_channel.astype(np.float32)
#img_channel /= img_channel.max()  # or use a fixed value if you know the max
tf.imshow(rescaled_stitch[:,:,0]+rescaled_stitch[:,:,1], cmap="gray")

In [ ]:
def preprocess_stitch_with_channel_third_dimension(stitched_img):
    preprocessed_stitch = np.stack([np.zeros_like(stitched_img[0]), np.zeros_like(stitched_img[0])], axis=-1)  

    for i, channel in enumerate(stitched_img):
        img_channel = stitched_img[i]
        if img_channel is None:
            print(f"Channel {i} is None, skipping.")
            continue
        if i == 1:
            prev_img_channel = stitched_img[0]
            open_imgs = [prev_img_channel, img_channel]
            for i, ch in enumerate(open_imgs):
                ch = img_01_normalization(ch)
                ch = exposure.equalize_adapthist(ch, kernel_size=100, clip_limit=0.05)
                ch = filters.gaussian(ch, sigma=2)
                open_imgs[i] = ch
            combined_channels = open_imgs[0] + (2*open_imgs[1]) 
            preprocessed_stitch[:,:,0] = combined_channels
        if i == 3:
            img_channel = img_01_normalization(img_channel)
            img_channel = exposure.equalize_adapthist(img_channel, kernel_size=100, clip_limit=0.05)
            img_channel = filters.gaussian(img_channel, sigma=2)
            preprocessed_stitch[:,:,1] = img_channel
    
    print("Preprocessed stitched image shape:", preprocessed_stitch.shape)
    rescaled_stitch = img_rescaled(preprocessed_stitch, factor=0.3)
    #tf.imshow((rescaled_stitch [:,:,0] + rescaled_stitch [:,:,1]))
    return rescaled_stitch

In [ ]:
def segement_cell_stitched_image(stitched_image, flow_threshold = 0.5, cellprob_threshold = -1, dilatemasks = True, plots = True, show = True, annotate=False, saveplots=False):

    tile_norm_blocksize = 0
    diameter = 180
    iter = 500
    min_size = 1000 #in pixels
    max_size_fraction = 0.1 #cells bigger than this fraction of the image are discarded

    masks, flows, styles = model.eval(stitched_image, batch_size=64, diameter=diameter, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold,
                                    normalize={"tile_norm_blocksize": tile_norm_blocksize},
                                    max_size_fraction=max_size_fraction,
                                    min_size=min_size,
                                    niter=iter)
    #dilate the masks to remove the jagged edges and fill holes
    if dilatemasks:
        masks = utils.dilate_masks(masks, n_iter=5)
#plot if true
    if plots:
        fig = plt.figure(figsize=(48,20),dpi=300)
        
        plot.show_segmentation(fig, stitched_image, masks, flows[0],channels=[0,1])
        if annotate:
            location = stitched_files[index].parent.parent.parent.name
            plt.annotate(location, (0,0), fontsize=20, color='dodgerblue', weight='bold')
        plt.tight_layout()
        if saveplots:
            maskdir = stitched_files[index].parent / "masks"
            maskdir.mkdir(exist_ok=True)
            plt.savefig(maskdir / f"{stitched_files[index].stem}_segmentationfigure.png", dpi=300, bbox_inches='tight')
        if show:
            plt.show()
    return masks

masks = segement_cell_stitched_image(rescaled_stitch, annotate=True, saveplots=True)

def old_segment_cell_stitchedimg(img, show=True):
    from skimage import exposure,filters,morphology
    gfp = img[:,:,0] #combine the ch1 and ch2 images to help cellpose out a bit
    rfp = img[:,:,1]
    dapi = img[:,:,2] #save ch3 for later
    
    img_combo = gfp+rfp
    img_combo = img_01_normalization(img_combo) #normalize to match cellpose training data
    #adjust contrast
    img_combo = exposure.equalize_adapthist(img_combo, kernel_size=68, clip_limit=0.01)
    #smooth and subtract background
    dog = filters.difference_of_gaussians(img_combo, low_sigma=2.5)
    img_combo = img_combo - dog
    
    #sharpen image and improve outline
    img_combo = filters.unsharp_mask(img_combo, radius=2, amount=1)
    
    #stack the images
    img_selected_channels = np.stack([img_combo, dapi],axis=-1)
    
    """ fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12,5),sharex=True, sharey=True)
    ax1.imshow(img_selected_channels[:,:,0])
    ax2.imshow(img_selected_channels[:,:,1]) """
     
    flow_threshold = 0.6
    cellprob_threshold = -1
    tile_norm_blocksize = 0
    diameter = 40

    masks, flows, styles = model.eval(img_selected_channels, batch_size=32, diameter=diameter, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold,
                                    normalize={"tile_norm_blocksize": tile_norm_blocksize})
    #plot if true
    if show:
        fig = plt.figure(figsize=(12,5))
        plot.show_segmentation(fig, img_selected_channels, masks, flows[0])
        plt.tight_layout()
        plt.show()
    return masks
    
def segment_nuclei_tweaking_stitchedimg(orig_img, show=True):
    from skimage import morphology, filters
    img = orig_img[:,:,2] # get the DAPI channel
    
    # remove background
    dog = filters.difference_of_gaussians(img, low_sigma=2.5)
    seed = np.minimum(dog, img)  # ensure seed is not greater than the original image
    bg = morphology.reconstruction(seed, img, method='dilation')
    img = img - bg
    
    # remove speckle-shaped autofluor
    bg2 = morphology.white_tophat(img, morphology.disk(3))
    img = img - bg2
    img = morphology.closing(img, morphology.disk(2.5))
    img = filters.gaussian(img, sigma=1)
    #img = img_01_normalization(img)
    
    flow_threshold = 0.5
    cellprob_threshold = 0
    tile_norm_blocksize = 0
    diameter = None

    masks, flows, styles = model.eval(img, batch_size=32, diameter=diameter, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold,
                                    normalize={"tile_norm_blocksize": tile_norm_blocksize})
    if show:
        fig = plt.figure(figsize=(24,6))
        plot.show_segmentation(fig, img, masks, flows[0])
        plt.tight_layout()
        plt.show()
    return masks


In [ ]:
import re
def find_row_col(well_code):
        rowcol_pattern = r"r(\d{1,2})c(\d{1,2})"  # Matches "RX" where X is the replicate number (placeholder for now)
        match = re.search(rowcol_pattern, well_code)
        if match:
            row_metadata = int(match.group(1))
            col_metadata = int(match.group(2))
        else:
            row_metadata = None 
            col_metadata = None
        return row_metadata,col_metadata
def get_masks_and_regionprops(masks, image, inputdir, outdir, removed_edge_masks=False, rescale_factor = 0.25):
    """Calculate region properties from masks and save them to a DataFrame.
    Args:
        masks (_type_): _description_
        image (_type_): _description_
        inputdir (_type_): _description_
        outdir (_type_): _description_
        removed_edge_masks (bool, optional): _description_. Defaults to False.
        rescale_factor (float, optional): _description_. Defaults to 0.25.
    """    
    from skimage import measure
    split_name = inputdir.name.split('_')
    savename =split_name[0]
    block = split_name[1].split('.')[0][-1]
    
    if removed_edge_masks:
        masks = utils.remove_edge_masks(masks, change_index=True)
        savename = savename+ "_removed_edges"
    props = measure.regionprops_table(
        masks, intensity_image=image,
        properties=('area','area_bbox','perimeter', 'intensity_mean','axis_major_length', 'axis_minor_length', 'centroid', 'orientation', 'eccentricity'),
    )
    props_df = pd.DataFrame(props)
    props_df.index.name = 'ObjectNumber'
    
    replicate_pattern = r"R(\d{1})"  # Matches "RX" where X is the replicate number (placeholder for now)
    match = re.search(replicate_pattern, inputdir.root)
    if match:
        replicate = int(match.group(1))
    else:
        replicate = None
    
    #rescale the area by a factor of 16 (inverse of 0.25^2)
    inverse_area_rescale_factor = 1 / (rescale_factor**2)
    print(inverse_area_rescale_factor)
    props_df["Cell_AreaShape_Area"] = props_df.apply(lambda x: x["area"]*inverse_area_rescale_factor, axis=1)
    
    props_df['Replicate_Number'] = replicate
    props_df['Filename'] = inputdir.name
    props_df['Parent_Folder'] = inputdir.parent.name
    props_df['Path'] = inputdir
    #print(savename)
    well_id = savename[:6]
    row, col = find_row_col(well_id)
    props_df['Metadata_Well_ID'] = well_id
    props_df[["Metadata_WellRow"]] = row
    props_df[["Metadata_WellColumn"]] = col
    # props_df["Metadata_WellRow"] = re.search(rowcol_pattern,savename).group(1),  # Assuming row is the next
    # props_df["Metadata_WellColumn"] = re.search(rowcol_pattern,savename).group(2)
    props_df["Metadata_Well"] = f"{chr(ord('@') + int(row))}{int(col):02d}"
    props_df['Block'] = block
# For rescale_factor = 0.25, use original_area = measured_area * 
    save_masks(savename, masks, outdir=outdir, image_ext='.tif')
    print(f"Saved masks to: {outdir}")

    #display(props_df)
    return props_df
outdir = Path("/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/stitching/segmentation_testset")
cell_df = get_masks_and_regionprops(masks, rescaled_stitch, stitched_files[0], outdir=maskdir)

cell_df_edges_removed = get_masks_and_regionprops(masks, rescaled_stitch, stitched_files[0], outdir=outdir, removed_edge_masks=True)



## Run Cellpose-SAM on folder of images

if you have many large images, you may want to run them as a loop over images
See `cellpose_functions` to run this
- moved to `save_mask_folder() `

In [ ]:
all_props = []
all_props_with_excluded = []

for i, img_path in enumerate(stitched_files):
    # Load image
    img = io.imread(img_path)
    # Select channels (use your selection logic, here using selected_channels)
    selected_img = select_channels(img)
    # Preprocess
    preprocessed_img = preprocess_stitched_img(selected_img)
    rescaled_img = img_rescaled(preprocessed_img, factor=0.25)
    # Segment
    masks = segement_cell_stitched_image(rescaled_img, show=False, annotate=True, saveplots=True)
    # Regionprops
    props_df = get_masks_and_regionprops(masks, rescaled_img, img_path, outdir=img_path.parent / "masks")
    props_df_with_excluded = get_masks_and_regionprops(masks, rescaled_img, img_path, outdir=img_path.parent / "masks", removed_edge_masks=True)
    # Add filename and metadata
    props_df['ImageNumber'] = i
    props_df_with_excluded['ImageNumber'] = i
    
    all_props.append(props_df)
    all_props_with_excluded.append(props_df_with_excluded)
    

# Combine all dataframes
combined_df = pd.concat(all_props, ignore_index=True)
combined_df_with_excluded_borders = pd.concat(all_props_with_excluded, ignore_index=True)
display(combined_df)

combined_df.to_csv(Path.joinpath(stitched_path,"stitched_test_data_v2.csv"))
combined_df.to_csv(Path.joinpath(stitched_path,"stitched_test_data_v2_borders_excluded.csv"))

### loop for all files in the group in the directory
```python
for i in trange(len(grouped_files)):
    file_group = grouped_files[i]
    img_set = load_image_set(file_group)
    img_set_name = get_image_set_name(file_group)
    print("Set name: ", set_name)
    
    stacked_img = img_preprocessing(img_set)
    rescaled_img = img_rescaled(stacked_img, factor=0.25)
    
    cell_masks = segment_cell(rescaled_img, show=False)
    nuc_masks = segment_nuclei(rescaled_img, show=False) 
    
    save_masks(img_set_name, cell_masks, image_ext=image_ext)
    save_masks(img_set_name, nuc_masks, image_ext=image_ext, mask_type="nuclei") 
```